# Firn Map Validation Against Historical Orthophotos

Brief, supplementary validation of the modelled firn extent (`firn_change.pro` output, see notebook 7) against independent high-resolution imagery: swisstopo's SWISSIMAGE "Zeitreise" (time-travel) orthophoto archive, which serves a historical flight-year mosaic (1949-present, sub-metre resolution) via a time-enabled WMS layer (`ch.swisstopo.swissimage-product`).

**Method**
1. For each of the 6 study glaciers, download the historical orthophoto for each year that also has a modelled firn snapshot (`yrout` in notebook 7 / `firn_change.pro`).
2. Flag years affected by fresh/residual snow on the surrounding bare terrain (which would make the glacier's snow cover unrepresentative of end-of-summer firn extent) using an automatic brightness heuristic on a buffer ring around the glacier outline, then visually confirm/override.
3. For the remaining "suitable" (glacier, year) pairs, manually digitize the visible firn/perennial-snow boundary on the orthophoto and rasterize it onto the same 10 m grid used by the firn model.
4. Compare the digitized firn cover fraction against the modelled firn cover fraction, per glacier and year.

**Why manual digitization, not an automated classifier**: an earlier version of this notebook classified firn/snow vs. bare ice with a global Otsu brightness threshold. That did not work here: at these small, very high-elevation, clean cirque glaciers, exposed bare ice is often optically about as bright as firn in true-colour imagery (no NIR/SWIR band is available in this archive to disambiguate them), so a global brightness split just traced the whole visually-bright glacier surface regardless of what the model predicted (mean disagreement ~39 percentage points, ~50% per-cell agreement - i.e. noise). Manual digitization lets a human eye use texture, shading, and local contrast cues that a single global threshold cannot.

**Caveat carried over regardless of method**: the WMS "Zeitreise" layer serves the most recent flight *up to and including* the requested year for tiles not reflown that year (regional flights follow a ~3-yearly cycle), so the true acquisition year can differ by up to ~2 years from the requested one - treat `year` as nominal, not an exact acquisition date.

Outputs: `figures/supplement/figS12_firn_validation_maps.pdf` and `figS13_firn_validation_summary.pdf` (grouped with the other firn/mass-balance supplementary figures, S09-S11).

## Imports

In [ ]:
%matplotlib inline
import matplotlib as mpl
mpl.rcParams['figure.dpi']   = 200
mpl.rcParams['font.family']  = 'Arial'

import os
import sys
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from pathlib import Path
from affine import Affine

import rasterio
from rasterio.features import geometry_mask
from rasterio.warp import reproject, Resampling
import geopandas as gpd
from shapely.ops import unary_union
from shapely.validation import make_valid
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.gpr_processing as gpr
import src.gpr_plotting as gprp
from src.geodata_processing import read_xyzn_to_gdf, add_panel_outline
from src.thermistor_plotting import build_profile_color_map

### Paths and configuration

In [ ]:
# --- Input / output directories ---
firn_dir   = project_root + '/results/firn_grids/'
xyzn_dir   = project_root + '/data/raw/sgi_2022/xyzn_lv95/'
bh_csv     = os.path.join(project_root, "data", "borehole_settings", "thermistor_coordinates.csv")
ortho_dir  = project_root + '/products/figures/firn_validation/orthophotos_zeitreise/'
output_dir = project_root + '/products/figures/firn_validation/'
os.makedirs(ortho_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# --- Glacier configuration (same 6 study glaciers / outlines as notebook 7) ---
glaciers = [
    {'key': 'alphubel',  'label': 'Alphubel',  'abbr': 'AH',
     'xyzn_file': 'SGI_2023_B55-15_lv95.xyzn',
     'bh_ids': ['AH1G', 'AH2G', 'AH3G', 'AH1TT', 'AH2TT', 'AH3TT']},
    {'key': 'felskinn',  'label': 'Chessjen',  'abbr': 'CJ',
     'xyzn_file': 'SGI_2023_B53-14_lv95.xyzn',
     'bh_ids': ['CJ1G', 'CJ2G', 'CJ1TT', 'CJ2TT', 'CJ3TT', 'CJ4TT']},
    {'key': 'hohsaas',   'label': 'Hohsaas',   'abbr': 'HS',
     'xyzn_file': 'SGI_2023_B51-13_lv95.xyzn',
     'bh_ids': ['HS1G', 'HS2G', 'HS3G', 'HS1TT', 'HS2TT', 'HS3TT']},
    {'key': 'sexrouge',  'label': 'Sex Rouge', 'abbr': 'SR',
     'xyzn_file': 'SGI_2023_B16-01_lv95.xyzn',
     'bh_ids': ['SR1TT', 'SR2TT']},
    {'key': 'tortin',    'label': 'Tortin',    'abbr': 'GT',
     'xyzn_file': 'SGI_2023_B75-12_lv95.xyzn',
     'bh_ids': ['GT1TT', 'GT2TT']},
    {'key': 'corvatsch', 'label': 'Corvatsch', 'abbr': 'CV',
     'xyzn_file': 'SGI_2022_E23-18_lv95.xyzn',
     'bh_ids': ['CV1TT', 'CV2TT']},
]

# Years with a modelled firn snapshot (firn_change.pro `yrout`), excluding 2025
# (the running "current" year - not tied to a single flight epoch).
VALIDATION_YEARS = [1980, 1990, 2000, 2010, 2014, 2019, 2022, 2024]

BBOX_BUFFER_M = 150   # extra margin around the glacier outline (m) - gives room for the
                       # "surrounding terrain" ring used in the fresh-snow heuristic below
PIXEL_SIZE_M  = 0.5   # orthophoto download resolution (m/px)
RING_WIDTH_M  = 150   # width of the bare-terrain ring around the outline used to flag fresh snow
BRIGHT_LUM    = 200   # luminance (0-255) above which a pixel counts as "bright" (snow-covered)
SNOW_RING_FRAC_THRESH = 0.30  # fraction of bright ring pixels above which a year is auto-flagged

ANNO_FONTSIZE = 12
ABBR_FONTSIZE = 12

### Load glacier outlines and modelled firn grids

Reuses the same `.grid` reader and snapshot years (`firn{yr}_{key}.grid`) as notebook 7.

In [ ]:
def read_arc_grid(path):
    """Read an Arc ASCII (.grid) file → (data, affine transform, tight bbox). Same as notebook 7."""
    with open(path) as f:
        ncols     = int(  f.readline().split()[1])
        nrows     = int(  f.readline().split()[1])
        xllcorner = float(f.readline().split()[1])
        yllcorner = float(f.readline().split()[1])
        cellsize  = float(f.readline().split()[1])
        nodata    = float(f.readline().split()[1])
        data = np.array(f.read().split(), dtype=np.float32).reshape(nrows, ncols)
    data[data == nodata] = np.nan
    if xllcorner < 1_000_000:  # LV03 -> LV95
        xllcorner += 2_000_000
        yllcorner += 1_000_000
    transform = Affine(cellsize, 0, xllcorner, 0, -cellsize, yllcorner + nrows * cellsize)
    valid = ~np.isnan(data)
    valid_rows = np.where(np.any(valid, axis=1))[0]
    valid_cols = np.where(np.any(valid, axis=0))[0]
    north = yllcorner + nrows * cellsize
    if len(valid_rows) and len(valid_cols):
        xmin = xllcorner + valid_cols[0]  * cellsize
        xmax = xllcorner + (valid_cols[-1] + 1) * cellsize
        ymax = north     - valid_rows[0]  * cellsize
        ymin = north     - (valid_rows[-1] + 1) * cellsize
    else:
        xmin, xmax, ymin, ymax = xllcorner, xllcorner + ncols * cellsize, yllcorner, yllcorner + nrows * cellsize
    return data, transform, (xmin, ymin, xmax, ymax)


def clean_outline_geom(geometries):
    """Dissolve outline parts into one polygonal geometry, repairing self-intersections.

    The .xyzn-derived outlines occasionally have self-intersecting rings, which makes
    shapely.difference() raise a GEOSException ("side location conflict"). make_valid() is
    the standard fix; it can return a GeometryCollection with stray points/lines mixed in,
    so only the polygonal parts are kept.
    """
    geom = unary_union(list(geometries))
    if not geom.is_valid:
        geom = make_valid(geom)
    if isinstance(geom, GeometryCollection):
        polys = [part for part in geom.geoms if isinstance(part, (Polygon, MultiPolygon))]
        geom = unary_union(polys) if polys else geom.buffer(0)
    return geom


DISPLAY_BUFFER_M = 50  # small padding around the model's own extent, for Figure S12 framing

for g in glaciers:
    k = g['key']

    # --- outline + bbox (bbox padded so the fresh-snow ring around the outline is fully covered) ---
    g['outline']      = read_xyzn_to_gdf(os.path.join(xyzn_dir, g['xyzn_file']))
    g['outline_geom'] = clean_outline_geom(g['outline'].geometry)
    g['bbox'] = tuple(gpr.square_bbox_from_gdf(
        g['outline'], buffer_m=BBOX_BUFFER_M + RING_WIDTH_M, pixel_size=PIXEL_SIZE_M))

    # --- modelled firn snapshots + reference (2025) glacier mask ---
    g['firnthick'], g['tfm'], model_bbox = read_arc_grid(firn_dir + f'firnthick_{k}.grid')
    g['glacier_mask_model'] = ~np.isnan(g['firnthick'])

    # tight display bbox from the model's own valid-cell extent, not the .xyzn outline: some
    # .xyzn outlines (e.g. hohsaas) are digitized as one continuous polygon covering more than
    # our study glacier (hohsaas's includes neighbouring Triftgletscher), which would otherwise
    # zoom Figure S12 out far past the glacier we actually model.
    g['bbox_display'] = tuple(gpr.make_square_bbox(model_bbox, buffer_m=DISPLAY_BUFFER_M))

    g['snapshots'] = {}
    for yr in VALIDATION_YEARS:
        path = firn_dir + f'firn{yr}_{k}.grid'
        if os.path.exists(path):
            snap, _, _ = read_arc_grid(path)
            g['snapshots'][yr] = snap

    # --- boreholes, for spatial context only ---
    if g['bh_ids'] and os.path.exists(bh_csv):
        g['boreholes'], _ = gpr.load_borehole_positions(bh_csv, keep_names=g['bh_ids'])
    else:
        g['boreholes'] = gpd.GeoDataFrame(columns=['geometry'], geometry='geometry', crs='EPSG:2056')

    print(f"{g['label']:12s}  model grid {g['firnthick'].shape}  "
          f"snapshots: {sorted(g['snapshots'].keys())}  bh: {len(g['boreholes'])}")

---
## Step 1 — Download SWISSIMAGE "Zeitreise" orthophotos

One image per (glacier, year); cached to disk so re-running the notebook doesn't re-hit the WMS.
`download_swisstopo_orthophoto_timetravel()` (added to `gpr_processing.py`) is the same WMS `GetMap`
call as the existing `download_swisstopo_orthophoto()` helper used elsewhere in this repo, with a
`TIME` parameter added and pointed at the historical `ch.swisstopo.swissimage-product` layer instead
of the current-day `ch.swisstopo.swissimage` mosaic.

In [ ]:
for g in glaciers:
    g['ortho_paths'] = {}
    for yr in VALIDATION_YEARS:
        out_tif = os.path.join(ortho_dir, f"{g['key']}_{yr}.tif")
        if not os.path.exists(out_tif):
            try:
                gpr.download_swisstopo_orthophoto_timetravel(
                    g['bbox'], out_tif, yr, pixel_size=PIXEL_SIZE_M)
            except Exception as e:
                print(f"  FAILED  {g['key']:10s} {yr}: {e}")
                continue
        g['ortho_paths'][yr] = out_tif
    print(f"{g['label']:12s}  {len(g['ortho_paths'])}/{len(VALIDATION_YEARS)} years downloaded")

---
## Step 2 — Screen years for fresh/residual snow

A year is only usable for firn-line validation if the imagery shows genuine end-of-summer conditions:
seasonal snow gone, only firn/perennial snow and bare ice remaining. There's no automated way to be
sure of that from RGB alone, so this is a two-stage check:

1. **Automatic heuristic** - within a bare-terrain ring around each glacier outline (rock/moraine that
   should never carry firn), compute the fraction of "bright" (likely snow-covered) pixels. A high
   fraction means the surroundings had fresh/residual snow at the time of the flight, so the glacier's
   own bright extent that year is not a reliable firn signal either → auto-flagged `fresh snow`.
2. **Manual override** - the heuristic is a coarse proxy (illumination, shadow, and scree brightness all
   confound it), so inspect the thumbnail grid below and fill in `MANUAL_OVERRIDE` to correct any
   mis-flagged (glacier, year) pairs before Step 3 uses `final_flag`.

In [ ]:
def read_luminance(tif_path):
    """RGB GeoTIFF -> (luminance array, affine transform)."""
    with rasterio.open(tif_path) as src:
        arr = src.read().astype(np.float32)  # (3, H, W)
        transform = src.transform
    lum = 0.2126 * arr[0] + 0.7152 * arr[1] + 0.0722 * arr[2]
    return lum, transform


rows = []
for g in glaciers:
    ring_geom = g['outline_geom'].buffer(RING_WIDTH_M).difference(g['outline_geom'])
    for yr, path in g['ortho_paths'].items():
        lum, tfm = read_luminance(path)
        ring_mask = geometry_mask([ring_geom], out_shape=lum.shape, transform=tfm, invert=True)
        if ring_mask.sum() == 0:
            continue
        bright_frac = float((lum[ring_mask] > BRIGHT_LUM).mean())
        auto_flag = 'fresh snow' if bright_frac > SNOW_RING_FRAC_THRESH else 'suitable'
        rows.append({'glacier': g['key'], 'label': g['label'], 'year': yr,
                     'ring_bright_frac': round(bright_frac, 3), 'auto_flag': auto_flag})

df_screen = pd.DataFrame(rows)
display(df_screen.pivot(index='label', columns='year', values='ring_bright_frac'))
print('\nauto-flagged fresh snow:',
      [(r.glacier, r.year) for r in df_screen.itertuples() if r.auto_flag == 'fresh snow'])

In [ ]:
# QC grid: one row per glacier, one column per year. Border colour = auto_flag.
# Use this to populate MANUAL_OVERRIDE in the next cell.
ncols = len(VALIDATION_YEARS)
nrows = len(glaciers)
fig, axs = plt.subplots(nrows, ncols, figsize=(2.0 * ncols, 2.0 * nrows), dpi=150)
fig.subplots_adjust(left=0.05, right=0.99, top=0.95, bottom=0.02, wspace=0.05, hspace=0.15)

flag_lookup = {(r.glacier, r.year): r.auto_flag for r in df_screen.itertuples()}

for ri, g in enumerate(glaciers):
    for ci, yr in enumerate(VALIDATION_YEARS):
        ax = axs[ri, ci]
        ax.set_xticks([]); ax.set_yticks([])
        if yr in g['ortho_paths']:
            with rasterio.open(g['ortho_paths'][yr]) as src:
                img = np.moveaxis(src.read(), 0, -1)
            ax.imshow(img)
            flag = flag_lookup.get((g['key'], yr), 'n/a')
            color = 'crimson' if flag == 'fresh snow' else 'seagreen'
            for spine in ax.spines.values():
                spine.set_edgecolor(color); spine.set_linewidth(3)
        else:
            ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes)
        if ri == 0:
            ax.set_title(str(yr), fontsize=10, fontweight='bold')
        if ci == 0:
            ax.set_ylabel(g['label'], fontsize=10, fontweight='bold')

legend_handles = [Patch(facecolor='none', edgecolor='seagreen', linewidth=3, label='auto: suitable'),
                  Patch(facecolor='none', edgecolor='crimson', linewidth=3, label='auto: fresh snow')]
fig.legend(handles=legend_handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.01), frameon=False)
# PNG, not PDF: this is a disposable QC contact sheet (not a manuscript figure), and PNG raster
# encoding sidesteps a PDF-backend font-embedding write that reliably timed out on this many panels
# when the project directory lives on an actively-syncing iCloud Drive.
plt.savefig(output_dir + 'qc_thumbnail_grid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Manual overrides after visually inspecting the thumbnail grid above (site-specific knowledge).
# Key: (glacier_key, year) -> 'suitable' | 'fresh snow'. Only entries that disagree with auto_flag
# need to be listed.
#
# The 5 'fresh snow' entries below passed the automatic ring heuristic (surrounding bare terrain
# looked clear) but were rejected during digitizing: the glacier surface itself had too much snow
# cover to identify a firn/ice boundary by eye. The ring check only looks at off-glacier terrain,
# so it can't catch this - hence the manual override.
#
# sexrouge/1980 is the opposite case: auto-flagged 'fresh snow' (ring_bright_frac=0.805, the
# surrounding rock looked snow-covered) but confirmed usable on the glacier itself when digitizing.
MANUAL_OVERRIDE = {
    ('alphubel', 2019): 'fresh snow',
    ('felskinn', 2019): 'fresh snow',
    ('felskinn', 2024): 'fresh snow',
    ('hohsaas', 2019):  'fresh snow',
    ('sexrouge', 1990): 'fresh snow',
    ('sexrouge', 1980): 'suitable',
}

df_screen['final_flag'] = [
    MANUAL_OVERRIDE.get((r.glacier, r.year), r.auto_flag) for r in df_screen.itertuples()
]

suitable = {(r.glacier, r.year) for r in df_screen.itertuples() if r.final_flag == 'suitable'}
excluded = {(r.glacier, r.year) for r in df_screen.itertuples() if r.final_flag == 'fresh snow'}
print(f'{len(suitable)} suitable (glacier, year) pairs, {len(excluded)} excluded for fresh/residual snow')
display(df_screen[df_screen['final_flag'] == 'fresh snow'][['label', 'year', 'ring_bright_frac']])

---
## Step 3 — Manually digitize the firn line

For each `suitable` (glacier, year) pair, trace the visible firn/perennial-snow boundary by eye
directly on the orthophoto. This requires a live, interactive Jupyter session (it can't be run
non-interactively) and the `ipympl` package (`%matplotlib widget` backend; added to `environment.yml`).

**Workflow:**
1. Run the `%matplotlib widget` cell below once.
2. Call `digitize('glacier_key', year)` - opens an interactive figure with the orthophoto and the
   modelled firn extent shown as a faint blue overlay for reference only (don't just trace the
   overlay - look at the actual image texture/shading). Each call starts fresh for that pair, so
   calling it again (e.g. to redo a mistake) is always safe.
3. Click vertices around the *area* that is firn/perennial-snow covered (not just a line marking
   where the boundary is - the polygon needs to enclose the full extent, since the comparison is
   area-based). To finish, click on the **first vertex again** to close the polygon (matplotlib's
   `PolygonSelector`, not a keyboard shortcut - there's no "press Enter" to finish). Useful extras:
   drag a vertex to move it (hold `ctrl` while the polygon is still open), right-click a vertex to
   delete it, `shift`+drag to move the whole polygon, `esc` to scrap it and start over.
   - **Disconnected firn areas** (e.g. felskinn's upper and lower parts, separated by bare ice):
     finish tracing one part, then call `digitize('glacier_key', year, new_part=True)` for the
     *same* (glacier_key, year) to trace the next part - `new_part=True` is what adds to the pair
     instead of starting over, and already-finished parts are redrawn in orange so you can see
     them while placing the next one.
4. Run `save_digitized_polygon('glacier_key', year)` to write it to
   `data/firn_validation/manual_firnlines/{glacier_key}_{year}.geojson` (all parts saved together).
   It warns if any two parts overlap - real disconnected patches shouldn't, so that usually means
   `new_part=True` was used by mistake to redo a part rather than add a genuinely new one.
5. Repeat for the other pairs in the checklist below (add as many cells as you like - `digitize()` /
   `save_digitized_polygon()` can be called repeatedly). Already-digitized pairs are skipped safely -
   delete the `.geojson` file to redo one.
6. Once done, switch back with `%matplotlib inline` before running the figure cells further down.

In [ ]:
%matplotlib widget

In [ ]:
from matplotlib.widgets import PolygonSelector

MANUAL_FIRNLINE_DIR = os.path.join(project_root, 'data', 'firn_validation', 'manual_firnlines')
os.makedirs(MANUAL_FIRNLINE_DIR, exist_ok=True)

# (glacier_key, year) -> list of PolygonSelector, one per disconnected part. A list (not a single
# selector) so glaciers with disconnected areas (e.g. felskinn's upper and lower parts) can be
# traced as several separate polygons before saving - see digitize()/save_digitized_polygon().
_active_selectors = {}


def digitize(glacier_key, year, new_part=False):
    """Open an interactive figure to trace one connected firn/perennial-snow area for
    (glacier_key, year).

    By default, each call starts fresh for this (glacier_key, year) - discarding anything traced
    previously in this session for the same pair. This is what you want to redo/restart a pair.

    Click vertices around the area (a closed polygon, not just a boundary line - the comparison
    is area-based). Click the first vertex again to close the polygon.

    If the firn/snow cover is genuinely split into several disconnected areas (e.g. an upper and
    a lower part with bare ice in between), finish tracing one part, then call
    digitize(glacier_key, year, new_part=True) for the SAME (glacier_key, year) to add the next
    part instead of starting over - already-finished parts are redrawn in orange for reference.
    Call save_digitized_polygon(glacier_key, year) once all parts are done; it saves all of them
    together as one multi-feature file and warns if any two parts overlap (a strong sign you
    meant to redo a part, not add a new disconnected one).

    Requires the `%matplotlib widget` cell above to have been run first.
    """
    g = next(x for x in glaciers if x['key'] == glacier_key)
    if year not in g['ortho_paths']:
        raise ValueError(f'no orthophoto cached for {glacier_key} {year}')

    key = (glacier_key, year)
    if not new_part or key not in _active_selectors:
        _active_selectors[key] = []
    parts_so_far = _active_selectors[key]

    with rasterio.open(g['ortho_paths'][year]) as src:
        img = np.moveaxis(src.read(), 0, -1)
        b = src.bounds

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img, extent=(b.left, b.right, b.bottom, b.top))
    if year in g['snapshots']:
        model_masked = np.where(g['snapshots'][year] > 0, 1.0, np.nan)
        gprp.imshow_grid(ax, model_masked, g['tfm'], cmap=plt.cm.Blues, alpha=0.30, zorder=5)

    # redraw already-finished parts (orange) so a new disconnected part can be placed sensibly
    for prev in parts_so_far:
        if len(prev.verts) >= 3:
            closed = list(prev.verts) + [prev.verts[0]]
            xs_p, ys_p = zip(*closed)
            ax.plot(xs_p, ys_p, color='orange', linewidth=1.5, zorder=14)

    ax.set_aspect('equal', adjustable='box')
    part_note = f' (part {len(parts_so_far) + 1})' if parts_so_far else ''
    ax.set_title(
        f"{g['label']} {year}{part_note}\n"
        f"trace one connected firn/snow area, close by clicking the first vertex again.\n"
        f"digitize('{glacier_key}', {year}, new_part=True) for another disconnected part, or "
        f"save_digitized_polygon('{glacier_key}', {year}) when done.",
        fontsize=9)

    selector = PolygonSelector(ax, onselect=lambda verts: None, useblit=True)
    parts_so_far.append(selector)
    return fig


def save_digitized_polygon(glacier_key, year):
    """Save all part(s) traced via digitize(glacier_key, year) to disk as one GeoJSON file
    (one feature per disconnected part; incomplete/abandoned parts with <3 vertices are skipped).
    Warns if any two saved parts overlap - real disconnected firn patches shouldn't overlap, so
    this usually means new_part=True was used to redo a part rather than add a genuinely new one.
    """
    key = (glacier_key, year)
    selectors = _active_selectors.get(key, [])
    polys = []
    for sel in selectors:
        if len(sel.verts) < 3:
            continue
        poly = Polygon(sel.verts)
        if not poly.is_valid:
            poly = poly.buffer(0)
        polys.append(poly)
    if not polys:
        raise ValueError('No finished polygon for this (glacier, year) - call digitize() and '
                         'trace at least 3 vertices first.')

    for a, b in itertools.combinations(polys, 2):
        inter = a.intersection(b).area
        if inter > 0:
            frac = inter / min(a.area, b.area)
            print(f"  WARNING  two parts overlap ({frac:.0%} of the smaller one's area) - "
                  f"if you meant to redo one part rather than add a new disconnected one, "
                  f"call digitize('{glacier_key}', {year}) WITHOUT new_part=True to start over.")

    out_path = os.path.join(MANUAL_FIRNLINE_DIR, f'{glacier_key}_{year}.geojson')
    gpd.GeoDataFrame({'geometry': polys}, crs='EPSG:2056').to_file(out_path, driver='GeoJSON')
    n_verts = sum(len(sel.verts) for sel in selectors if len(sel.verts) >= 3)
    print(f"saved {out_path}  ({len(polys)} part{'s' if len(polys) != 1 else ''}, "
          f"{n_verts} vertices total)")


print('Checklist - suitable (glacier, year) pairs:')
for gl, yr in sorted(suitable):
    done = os.path.exists(os.path.join(MANUAL_FIRNLINE_DIR, f'{gl}_{yr}.geojson'))
    print(f"  [{'done' if done else 'TODO'}]  {gl:10s} {yr}")

Example for one pair (repeat in new cells for the rest of the checklist above):

```python
digitize('alphubel', 2024)
```
```python
save_digitized_polygon('alphubel', 2024)
```

Once every pair you want to include is saved, run the loader cell below (safe to re-run any time -
it just re-reads whatever `.geojson` files currently exist).

In [ ]:
%matplotlib inline

for g in glaciers:
    g['manual_polygon']   = {}   # year -> shapely geometry (for plotting the actual traced line)
    g['manual_firn_mask'] = {}   # year -> boolean array on the model's 10 m grid (for stats)
    for yr in g['ortho_paths']:
        if (g['key'], yr) not in suitable:
            continue
        path = os.path.join(MANUAL_FIRNLINE_DIR, f"{g['key']}_{yr}.geojson")
        if not os.path.exists(path):
            continue
        geom = unary_union(gpd.read_file(path).geometry)
        g['manual_polygon'][yr] = geom
        g['manual_firn_mask'][yr] = geometry_mask(
            [geom], out_shape=g['firnthick'].shape, transform=g['tfm'], invert=True)
    print(f"{g['label']:12s}  digitized years: {sorted(g['manual_firn_mask'].keys())}")

In [ ]:
digitize('hohsaas', 2022, new_part=True)

In [ ]:
save_digitized_polygon('hohsaas', 2022)

---
## Step 4 — Compare to the modelled firn extent

Per (glacier, year): modelled firn cover = fraction of the reference glacier mask with modelled firn
thickness > 0; digitized firn cover = fraction of the same mask inside the manually traced polygon.
`agreement_pct` is the fraction of cells where the two binary classifications agree (overall
accuracy).

In [ ]:
val_rows = []
for g in glaciers:
    ref_mask = g['glacier_mask_model']
    for yr, digitized_mask in g['manual_firn_mask'].items():
        if yr not in g['snapshots']:
            continue
        model_binary = g['snapshots'][yr] > 0
        n_cells = int(ref_mask.sum())
        if n_cells == 0:
            continue
        dig_binary = digitized_mask[ref_mask]
        mod_binary = model_binary[ref_mask]
        val_rows.append({
            'glacier': g['key'], 'label': g['label'], 'year': yr,
            'n_cells': n_cells,
            'model_firn_pct':      100 * mod_binary.mean(),
            'digitized_firn_pct':  100 * dig_binary.mean(),
            'agreement_pct':       100 * (mod_binary == dig_binary).mean(),
        })

# explicit `columns=` so df_val still has the expected columns (just 0 rows) before anything has
# been digitized yet - pd.DataFrame([]) alone gives a columnless frame and breaks every
# downstream df_val['col'] lookup (incl. in the Figure S13 cell) with a KeyError
VAL_COLUMNS = ['glacier', 'label', 'year', 'n_cells', 'model_firn_pct', 'digitized_firn_pct', 'agreement_pct']
df_val = pd.DataFrame(val_rows, columns=VAL_COLUMNS)
if len(df_val):
    df_val = df_val.sort_values(['label', 'year']).reset_index(drop=True)
df_val['abs_diff_pct'] = (df_val['model_firn_pct'] - df_val['digitized_firn_pct']).abs()
display(df_val.round(1))

print(f"\nn = {len(df_val)} (glacier, year) comparisons")
if len(df_val):
    print(f"mean |model - digitized| firn cover: {df_val['abs_diff_pct'].mean():.1f} pct. points")
    print(f"mean per-cell agreement: {df_val['agreement_pct'].mean():.1f} %")

---
## Figure S12 — Spatial validation maps

One panel per glacier: most recent digitized year, high-resolution orthophoto in the background,
modelled firn extent as a blue fill, manually digitized firn/snow boundary as a red line. Close
agreement between the fill edge and the red line is the visual validation signal.

In [ ]:
# Representative year per glacier = most recent one with a valid model/digitized comparison,
# unless overridden below.
best_year = df_val.sort_values('year').groupby('glacier')['year'].last().to_dict() if len(df_val) else {}

# Manually pick a different year to display for specific glaciers (key -> year). Only takes
# effect if that (glacier, year) actually has a Step 4 comparison; otherwise falls back to the
# auto-picked year with a warning, rather than silently showing nothing.
FIGURE_YEAR_OVERRIDE = {
    'hohsaas': 1990,
    'alphubel': 2022,
    'felskinn': 2000,
}
_val_pairs = set(zip(df_val['glacier'], df_val['year']))
for _gk, _yr in FIGURE_YEAR_OVERRIDE.items():
    if (_gk, _yr) in _val_pairs:
        best_year[_gk] = _yr
    else:
        print(f"  WARNING  FIGURE_YEAR_OVERRIDE: no comparison for ({_gk}, {_yr}) - "
              f"keeping auto-picked year {best_year.get(_gk)}")

cmap_firn  = plt.cm.Blues
norm_firn  = BoundaryNorm([0, 0.5, 1], 256, clip=True)
fmt = mticker.FuncFormatter(lambda v, _: f"{int(v):,}".replace(',', "'"))

SP = dict(left=0.09, right=0.99, top=0.94, bottom=0.14, wspace=0.08, hspace=0.12)
fig, axs = plt.subplots(2, 3, figsize=(11, 7.5), dpi=200)
fig.subplots_adjust(**SP)

for i, (g, ax) in enumerate(zip(glaciers, axs.flat)):
    yr = best_year.get(g['key'])
    x0, y0, x1, y1 = g['bbox_display']
    ax.set_facecolor('white')

    if yr is None:
        ax.text(0.5, 0.5, 'not yet digitized', ha='center', va='center', transform=ax.transAxes)
        ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect('equal', adjustable='box')
        ax.set_title(g['label'], fontsize=ANNO_FONTSIZE, fontweight='bold')
        continue

    with rasterio.open(g['ortho_paths'][yr]) as src:
        img = np.moveaxis(src.read(), 0, -1)
        b = src.bounds
    ax.imshow(img, extent=(b.left, b.right, b.bottom, b.top), zorder=1)

    model_masked = np.where(g['snapshots'][yr] > 0, 1.0, np.nan)
    gprp.imshow_grid(ax, model_masked, g['tfm'], cmap=cmap_firn, alpha=0.55, norm=norm_firn, zorder=10)

    manual_geom = g['manual_polygon'][yr]
    parts = manual_geom.geoms if hasattr(manual_geom, 'geoms') else [manual_geom]
    for part in parts:
        if part.is_empty:
            continue
        xs_p, ys_p = part.exterior.xy
        ax.plot(xs_p, ys_p, color='red', linewidth=1.8, zorder=15)

    if len(g['boreholes']) > 0:
        ax.scatter(g['boreholes'].geometry.x, g['boreholes'].geometry.y,
                   s=45, marker='o', facecolor='gold', edgecolor='black', linewidth=1.0, zorder=21)

    row = df_val[(df_val['glacier'] == g['key']) & (df_val['year'] == yr)].iloc[0]
    ax.text(0.03, 0.03, f"{g['label']}\n{yr}", transform=ax.transAxes,
            ha='left', va='bottom', fontsize=ANNO_FONTSIZE - 1, fontweight='bold', zorder=25,
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='none', alpha=0.85))
    ax.text(0.97, 0.97,
            f"model {row['model_firn_pct']:.0f}%  |  digitized {row['digitized_firn_pct']:.0f}%\n"
            f"agreement {row['agreement_pct']:.0f}%",
            transform=ax.transAxes, ha='right', va='top', fontsize=ANNO_FONTSIZE - 3, zorder=25,
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='none', alpha=0.85))

    add_panel_outline(ax, g['bbox_display'], color='black', linewidth=1.2)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect('equal', adjustable='box')

    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune=None))
    ax.xaxis.set_major_formatter(fmt)
    ax.tick_params(axis='x', labelsize=8)
    ax.set_xlabel('Easting (LV95) [m]' if i // 3 == 1 else '', fontsize=ANNO_FONTSIZE - 2)

    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune=None))
    ax.yaxis.set_major_formatter(fmt)
    ax.tick_params(axis='y', rotation=90, labelsize=8)
    ax.set_ylabel('Northing (LV95) [m]' if i % 3 == 0 else '', fontsize=ANNO_FONTSIZE - 2)

legend_handles = [
    Patch(facecolor=cmap_firn(0.6), edgecolor='none', alpha=0.55, label='Modelled firn extent'),
    Patch(facecolor='none', edgecolor='red', linewidth=1.8, label='Manually digitized firn line'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, 0.005),
          fontsize=ANNO_FONTSIZE - 2, frameon=False)

plt.savefig(os.path.join(project_root, 'figures', 'supplement', 'figS12_firn_validation_maps.pdf'),
           dpi=150, bbox_inches='tight')
plt.show()

---
## Figure S13 — Summary comparison across all digitized years

Modelled vs. manually digitized firn cover fraction for every (glacier, year) comparison,
colour-coded by glacier. Points on the 1:1 line indicate perfect agreement between the model and
the independent orthophoto-based digitization.

In [ ]:
glacier_labels = [g['label'] for g in glaciers]
COLORS = build_profile_color_map(glacier_labels)

fig, ax = plt.subplots(figsize=(6.5, 6.5), dpi=200)

for label in glacier_labels:
    sub = df_val[df_val['label'] == label]
    if sub.empty:
        continue
    ax.scatter(sub['model_firn_pct'], sub['digitized_firn_pct'], s=70,
              color=COLORS[label], edgecolor='black', linewidth=0.8, label=label, zorder=5)
    for _, r in sub.iterrows():
        ax.annotate(str(int(r['year'])), (r['model_firn_pct'], r['digitized_firn_pct']),
                   xytext=(4, 4), textcoords='offset points', fontsize=7)

lims = [0, 100]
ax.plot(lims, lims, 'k--', lw=1, zorder=1, label='1:1')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Modelled firn cover [% of glacier area]', fontsize=ANNO_FONTSIZE)
ax.set_ylabel('Manually digitized firn cover [% of glacier area]', fontsize=ANNO_FONTSIZE)
ax.grid(True, alpha=0.25, linestyle='--')
ax.legend(fontsize=ANNO_FONTSIZE - 4, ncol=2, framealpha=1.0, edgecolor='black', fancybox=False)

if len(df_val):
    bias = (df_val['digitized_firn_pct'] - df_val['model_firn_pct']).mean()
    mae  = df_val['abs_diff_pct'].mean()
    rmse = np.sqrt(((df_val['digitized_firn_pct'] - df_val['model_firn_pct']) ** 2).mean())
    # Pearson r^2 (co-variation between model and digitized values) - NOT the same as skill
    # relative to the 1:1 line (that's what bias/MAE/RMSE above already capture). r^2 stays high
    # even with a systematic offset or scale mismatch, so report both, not r^2 alone.
    r2 = np.nan if len(df_val) < 2 else np.corrcoef(df_val['model_firn_pct'], df_val['digitized_firn_pct'])[0, 1] ** 2
    ax.text(0.03, 0.97,
           f'n = {len(df_val)}\nR² = {r2:.2f}\nbias = {bias:+.1f} pct. pts\n'
           f'MAE = {mae:.1f} pct. pts\nRMSE = {rmse:.1f} pct. pts',
           transform=ax.transAxes, ha='left', va='top', fontsize=ANNO_FONTSIZE - 3,
           bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='black', alpha=0.9))

plt.savefig(os.path.join(project_root, 'figures', 'supplement', 'figS13_firn_validation_summary.pdf'),
           dpi=150, bbox_inches='tight')
plt.show()

#### Caption material for Figure S13

Not rendered on the figure itself - copy the relevant bit into the manuscript's figure caption.
The cell below prints the excluded-years list from whatever `df_screen`/`MANUAL_OVERRIDE` state was
used to produce the figure above, so it can't drift out of sync with a re-run.

In [ ]:
excl = df_screen[df_screen['final_flag'] == 'fresh snow']
if not excl.empty:
    excl_txt = '; '.join(f"{lbl}: {sorted(sub['year'])}" for lbl, sub in excl.groupby('label'))
    print(f'Excluded for fresh/residual snow cover - {excl_txt}')
else:
    print('No years excluded.')

---
## Compress output PDFs

Same Ghostscript pass used in notebook 7, kept here for consistency (repo convention: figures ≤ 2 MB).

In [ ]:
import subprocess, shutil

MAX_BYTES = 2 * 1024 * 1024  # 2 MB
GS_BIN = shutil.which('gs') or '/usr/local/bin/gs'
GS_SETTINGS = ['/printer', '/ebook', '/screen']

fig_files = [
    Path(project_root) / 'figures' / 'supplement' / 'figS12_firn_validation_maps.pdf',
    Path(project_root) / 'figures' / 'supplement' / 'figS13_firn_validation_summary.pdf',
]

def compress_pdf(path: Path, max_bytes: int = MAX_BYTES) -> None:
    original_bytes = path.stat().st_size
    if original_bytes <= max_bytes:
        print(f"  ok   {path.name}  ({original_bytes/1e6:.2f} MB)")
        return
    tmp = path.with_suffix('.compressed.pdf')
    for setting in GS_SETTINGS:
        result = subprocess.run([
            GS_BIN, '-dBATCH', '-dNOPAUSE', '-q',
            '-sDEVICE=pdfwrite', '-dCompatibilityLevel=1.4',
            f'-dPDFSETTINGS={setting}',
            f'-sOutputFile={tmp}', str(path)
        ], capture_output=True)
        if result.returncode != 0:
            print(f"  ERROR running gs on {path.name}: {result.stderr.decode()}")
            tmp.unlink(missing_ok=True)
            return
        compressed_bytes = tmp.stat().st_size
        if compressed_bytes <= max_bytes:
            shutil.move(tmp, path)
            print(f"  compressed  {path.name}: "
                  f"{original_bytes/1e6:.2f} MB → {compressed_bytes/1e6:.2f} MB  (gs {setting})")
            return
        tmp.unlink(missing_ok=True)
    print(f"  WARNING  {path.name}: still {compressed_bytes/1e6:.2f} MB after all gs settings")

for p in fig_files:
    if p.exists():
        compress_pdf(p)
    else:
        print(f"  not found: {p.name}")
print("\nDone.")